# Loading Data


In [123]:
from zipfile import compressor_names

import pandas as pd
import ast
import numpy as np

In [124]:
mbm_bat = pd.read_csv(r'D:\all_league\all_league_batting_card.csv')
mbm_bowl = pd.read_csv(r'D:\all_league\all_league_bowling_card.csv')
playing_xi = pd.read_csv(r'D:\all_league\all_league_playing_xi.csv')
player_info = pd.read_csv(r'D:\all_league\people3.csv')

# Cleaning Data

In [125]:
mbm_bat['batter_id'] = mbm_bat['batter_id'].apply(lambda x: str(int(float(x))) if pd.notnull(x) else x)

# Function to parse playing_xi columns
def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Define a helper function that returns the longest non-null string among the three names
def get_longest_name(row):
    names = [row['name'], row['unique_name']]
    # Filter out any missing values
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Ensure all player IDs in playing_xi are strings
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(lambda x: [str(pid) for pid in x])
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(lambda x: [str(pid) for pid in x])

# and the value is the longest name obtained by applying the function above
player_map = dict(zip(player_info['key_cricinfo'].astype(str), player_info.apply(get_longest_name, axis=1)))

standardized_player_map = {}
for key, value in player_map.items():
    clean_key = str(key).replace('.0', '')
    standardized_player_map[clean_key] = value

## Handling DNB Values in Batting Card

In [126]:
dnb_rows = []

# Group mbm_bat by match_id and innings (since a match can have two innings)
for (match_id, inns), group in mbm_bat.groupby(['match_id', 'inns']):
    # Get common details from the group (assumed to be the same for all rows in the group)
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]
    league = group['league'].iloc[0]  # Extract league from the group

    # Determine the set of batter IDs (as strings) who batted in this innings
    batted_ids = set(group['batter_id'].unique())

    # Lookup the corresponding playing_xi row for this match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]
    # Determine which playing XI list corresponds to the batting team
    if row['team1_name'] == bat_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bat_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Batting team {bat_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert all player IDs in the playing XI list to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in the playing XI who did not bat (i.e. DNB)
    dnb_ids = [pid for pid in full_xi if pid not in batted_ids]

    # For each DNB player, create a new row with default values
    for pid in dnb_ids:
        batter_name = player_map.get(pid, "")
        new_row = {
            'date': date,
            'match_id': match_id,
            'bat_team': bat_team,
            'bowl_team': bowl_team,
            'inns': inns,
            'venue': venue,
            'batter_id': pid,
            'batter': batter_name,
            'runs': 0,
            'balls': 0,
            'fours': 0,
            'sixes': 0,
            'sr': 0,
            'isOut': False,
            'wicketType': "DNB",
            'bowler_id': np.nan,
            'bowler': np.nan,
            'league': league  # Include league in the new row
        }
        dnb_rows.append(new_row)

In [127]:
# Create the combined dataframe
dnb_df = pd.DataFrame(dnb_rows)
combined_df = pd.concat([mbm_bat, dnb_df], ignore_index=True)

In [128]:
# Update batter names using standardized player map
combined_df['batter'] = combined_df.apply(
    lambda row: row['batter'] if pd.notnull(row['batter']) and row['batter'] != ""
    else standardized_player_map.get(str(row['batter_id']).replace('.0', ''), ""),
    axis=1
)

# Correctly identify DNB rows based on wicketType to sort
combined_df['is_DNB'] = combined_df['wicketType'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB'], inplace=True)

# Convert bowler_id float values to integer, preserving NaN values
combined_df['batter_id'] = combined_df['batter_id'].astype(int)
combined_df['bowler_id'] = combined_df['bowler_id'].apply(lambda x: int(float(x)) if pd.notnull(x) else x)

In [129]:
combined_df = combined_df.reindex(columns=['match_id', 'date', 'inns', 'batter', 'batter_id', 'bat_team', 'bowl_team', 'runs', 'balls', 'fours', 'sixes', 'sr', 'isOut', 'wicketType', 'bowler', 'bowler_id', 'venue', 'league'])

In [130]:
combined_df[combined_df.match_id == 1291205]

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,fours,sixes,sr,isOut,wicketType,bowler,bowler_id,venue,league
110545,1291205,2021-12-11,1,N Dickwella,429754,Dambulla Sixers,Colombo Strikers,30,16,6,0,187.50,True,caught,S Prasanna,268740.0,R.Premadasa Stadium,LPL
110546,1291205,2021-12-11,1,PD Salt,669365,Dambulla Sixers,Colombo Strikers,21,18,4,0,116.67,True,stumped,S Prasanna,268740.0,R.Premadasa Stadium,LPL
110547,1291205,2021-12-11,1,Sohaib Maqsood,43266,Dambulla Sixers,Colombo Strikers,24,22,4,0,109.09,True,lbw,JDF Vandersay,370040.0,R.Premadasa Stadium,LPL
110548,1291205,2021-12-11,1,KTH Ratnayake,950405,Dambulla Sixers,Colombo Strikers,5,15,0,0,33.33,True,caught,JDF Vandersay,370040.0,R.Premadasa Stadium,LPL
110549,1291205,2021-12-11,1,MD Shanaka,437316,Dambulla Sixers,Colombo Strikers,16,15,2,0,106.67,True,caught,R Rampaul,52912.0,R.Premadasa Stadium,LPL
110550,1291205,2021-12-11,1,Najibullah Zadran,524049,Dambulla Sixers,Colombo Strikers,11,15,1,0,73.33,True,lbw,JDF Vandersay,370040.0,R.Premadasa Stadium,LPL
110551,1291205,2021-12-11,1,RTM Mendis,629084,Dambulla Sixers,Colombo Strikers,8,8,0,0,100.00,True,caught,R Rampaul,52912.0,R.Premadasa Stadium,LPL
110552,1291205,2021-12-11,1,C Karunaratne,623695,Dambulla Sixers,Colombo Strikers,12,10,0,1,120.00,True,lbw,R Rampaul,52912.0,R.Premadasa Stadium,LPL
110553,1291205,2021-12-11,1,S Jayathilake,727399,Dambulla Sixers,Colombo Strikers,1,1,0,0,100.00,True,run out,NaN,NaN,R.Premadasa Stadium,LPL
110554,1291205,2021-12-11,1,Imran Tahir,40618,Dambulla Sixers,Colombo Strikers,0,0,0,0,0.00,False,DNB,NaN,NaN,R.Premadasa Stadium,LPL


In [131]:
combined_df.to_csv(r"D:\all_league\all_league_bat_final.csv")

## Handling DNB Values in Bowling Card

In [132]:
mbm_bowl = pd.read_csv(r'D:\all_league\all_league_bowling_card.csv')
playing_xi = pd.read_csv(r'D:\all_league\all_league_playing_xi.csv')
player_info = pd.read_csv(r'D:\all_league\people3.csv')

In [133]:
# Standardize bowler_id in mbm_bowl
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].astype(str).str.replace(r'\.0$', '', regex=True)

# Convert key_cricinfo to string
player_info['key_cricinfo'] = player_info['key_cricinfo'].astype(str)

# Standardize bowler_id in mbm_bowl
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].apply(lambda x: str(int(float(x))) if pd.notnull(x) else x)

# Function to parse playing_xi columns
def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Ensure all player IDs in playing_xi are strings
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(lambda x: [str(pid) for pid in x])
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(lambda x: [str(pid) for pid in x])

# Define a helper function that returns the longest non-null string among the names
def get_longest_name(row):
    names = [row['name'], row['unique_name']]
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Create the player mapping dictionary
player_map = dict(zip(player_info['key_cricinfo'], player_info.apply(get_longest_name, axis=1)))

In [134]:
# Standardize player_map keys for consistent format
standardized_player_map = {}
for key, value in player_map.items():
    clean_key = str(key).replace('.0', '')
    standardized_player_map[clean_key] = value

# Create a function to standardize player IDs consistently
def standardize_id(player_id):
    if pd.isna(player_id):
        return ""
    return str(player_id).replace('.0', '')

dnb_bowl_rows = []

# Group by match_id and innings
for (match_id, inns), group in mbm_bowl.groupby(['match_id', 'inns']):
    # Extract match details
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]
    league = group['league'].iloc[0]

    # Get set of bowler IDs who actually bowled (standardized format)
    bowled_ids = {standardize_id(bid) for bid in group['bowler_id'].unique()}

    # Lookup the playing XI row for the match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]

    # Get the playing XI for the bowling team
    if row['team1_name'] == bowl_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bowl_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Bowling team {bowl_team} not found in playing XI teams for match {match_id}.")
        continue

    # Find players who didn't bowl (in playing XI but not in bowled_ids)
    dnb_bowlers = [pid for pid in full_xi if standardize_id(pid) not in bowled_ids]

    # Add a new row for each bowler who did not bowl
    for pid in dnb_bowlers:
        # Get the bowler's name from the player map
        bowler_name = standardized_player_map.get(standardize_id(pid), "")

        new_row = {
            'date': date,
            'match_id': match_id,
            'bowl_team': bowl_team,
            'bat_team': bat_team,
            'bowler_id': pid,
            'bowler': bowler_name,
            'inns': inns,
            'overs': 'DNB',
            'runs': 0,
            'wickets': 0,
            'maidens': 0,
            'eco': 0,
            'balls': 0,
            'lbw_or_bowled': 0,
            'dots': 0,
            'venue': venue,
            'league': league  # Include league in the new row
        }
        dnb_bowl_rows.append(new_row)

# Make the DNB dataframe
dnb_df = pd.DataFrame(dnb_bowl_rows)
# Combine the original and DNB DataFrames
combined_df = pd.concat([mbm_bowl, dnb_df], ignore_index=True)

In [135]:
# Also update this section to use the standardized player map
combined_df['bowler'] = combined_df.apply(
    lambda row: row['bowler'] if pd.notnull(row['bowler']) and row['bowler'] != ""
    else standardized_player_map.get(str(row['bowler_id']).replace('.0', ''), ""),
    axis=1
)
# Sort and save the combined DataFrame
combined_df['is_DNB_Bowl'] = combined_df['overs'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB_Bowl']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB_Bowl'], inplace=True)

In [136]:
combined_df[combined_df.match_id == 1291205]

,match_id,bowler,bowler_id,bowl_team,bat_team,inns,overs,balls,wickets,runs,lbw_or_bowled,dots,eco,maidens,venue,date,league
110586,1291205,R Rampaul,52912,Colombo Strikers,Dambulla Sixers,1,4.0,25,3,27.0,1,12,6.750000,0,R.Premadasa Stadium,2021-12-11,LPL
110587,1291205,PVD Chameera,552152,Colombo Strikers,Dambulla Sixers,1,2.0,13,0,26.0,0,5,13.000000,0,R.Premadasa Stadium,2021-12-11,LPL
110588,1291205,S Prasanna,268740,Colombo Strikers,Dambulla Sixers,1,4.0,24,2,17.0,0,14,4.250000,0,R.Premadasa Stadium,2021-12-11,LPL
110589,1291205,DM de Silva,465793,Colombo Strikers,Dambulla Sixers,1,2.0,12,0,13.0,0,5,6.500000,0,R.Premadasa Stadium,2021-12-11,LPL
110590,1291205,Naveen-ul-Haq,793447,Colombo Strikers,Dambulla Sixers,1,3.0,19,0,15.0,0,11,5.000000,0,R.Premadasa Stadium,2021-12-11,LPL
110591,1291205,SMA Priyanjan,233802,Colombo Strikers,Dambulla Sixers,1,1.0,6,0,8.0,0,1,8.000000,0,R.Premadasa Stadium,2021-12-11,LPL
110592,1291205,JDF Vandersay,370040,Colombo Strikers,Dambulla Sixers,1,4.0,24,3,25.0,2,10,6.250000,0,R.Premadasa Stadium,2021-12-11,LPL
110593,1291205,LD Chandimal,300628,Colombo Strikers,Dambulla Sixers,1,DNB,0,0,0.0,0,0,0.000000,0,R.Premadasa Stadium,2021-12-11,LPL
110594,1291205,MDKJ Perera,300631,Colombo Strikers,Dambulla Sixers,1,DNB,0,0,0.0,0,0,0.000000,0,R.Premadasa Stadium,2021-12-11,LPL
110595,1291205,SE Rutherford,914541,Colombo Strikers,Dambulla Sixers,1,DNB,0,0,0.0,0,0,0.000000,0,R.Premadasa Stadium,2021-12-11,LPL


In [137]:
combined_df.columns

Index(['match_id', 'bowler', 'bowler_id', 'bowl_team', 'bat_team', 'inns',
       'overs', 'balls', 'wickets', 'runs', 'lbw_or_bowled', 'dots', 'eco',
       'maidens', 'venue', 'date', 'league'],
      dtype='object')

In [138]:
combined_df = combined_df.reindex(columns=['match_id', 'date', 'inns', 'bowler', 'bowler_id', 'bowl_team', 'bat_team', 'overs', 'runs', 'wickets', 'maidens', 'eco', 'balls', 'dots', 'lbw_or_bowled', 'venue', 'league'])

In [139]:
combined_df.to_csv(r"D:\all_league\all_league_bowl_final.csv")